# Sales Intelligence Dashboard - Data Analysis

**Objective:** Analyze sales data to identify 15% revenue growth opportunities worth $1.5M

## Analysis Sections:
1. Data Loading & Exploration
2. Customer Segmentation (RFM Analysis)
3. Customer Lifetime Value (CLV)
4. Churn Prediction
5. Revenue Opportunity Identification
6. Sales Performance Analysis
7. Export Processed Data for Tableau

In [5]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Data Loading & Exploration

In [6]:
# Load all datasets
customers_df = pd.read_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/raw/customers.csv')
products_df = pd.read_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/raw/products.csv')
sales_reps_df = pd.read_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/raw/sales_reps.csv')
transactions_df = pd.read_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/raw/transactions.csv')
# Convert date columns
customers_df['signup_date'] = pd.to_datetime(customers_df['signup_date'])
transactions_df['transaction_date'] = pd.to_datetime(
    transactions_df['transaction_date'])
sales_reps_df['hire_date'] = pd.to_datetime(sales_reps_df['hire_date'])

print("Data loaded successfully!")
print(f"\nCustomers: {len(customers_df):,}")
print(f"Products: {len(products_df):,}")
print(f"Sales Reps: {len(sales_reps_df):,}")
print(f"Transactions: {len(transactions_df):,}")

Data loaded successfully!

Customers: 10,000
Products: 25
Sales Reps: 15
Transactions: 50,000


In [7]:
# Filter for closed won transactions
won_transactions = transactions_df[transactions_df['is_won'] == True].copy()

# Calculate current revenue baseline
total_revenue = won_transactions['net_amount'].sum()
total_profit = won_transactions['profit'].sum()
avg_deal_size = won_transactions['net_amount'].mean()

print("="*60)
print("CURRENT REVENUE METRICS")
print("="*60)
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Average Deal Size: ${avg_deal_size:,.2f}")
print(f"Total Deals Won: {len(won_transactions):,}")
print(f"\nTarget Revenue (15% growth): ${total_revenue * 1.15:,.2f}")
print(f"Revenue Opportunity Gap: ${total_revenue * 0.15:,.2f}")

CURRENT REVENUE METRICS
Total Revenue: $753,689,690.34
Total Profit: $411,970,546.34
Average Deal Size: $30,290.56
Total Deals Won: 24,882

Target Revenue (15% growth): $866,743,143.89
Revenue Opportunity Gap: $113,053,453.55


## 2. Customer Segmentation (RFM Analysis)

In [8]:
# Calculate RFM metrics
analysis_date = won_transactions['transaction_date'].max() + timedelta(days=1)

rfm_df = won_transactions.groupby('customer_id').agg({
    'transaction_date': lambda x: (analysis_date - x.max()).days,
    'transaction_id': 'count',
    'net_amount': 'sum'
}).reset_index()

rfm_df.columns = ['customer_id', 'recency', 'frequency', 'monetary']

# Create RFM scores
rfm_df['R_score'] = pd.qcut(rfm_df['recency'], 5, labels=[5, 4, 3, 2, 1])
rfm_df['F_score'] = pd.qcut(rfm_df['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm_df['M_score'] = pd.qcut(rfm_df['monetary'], 5, labels=[1, 2, 3, 4, 5])

rfm_df['R_score'] = rfm_df['R_score'].astype(int)
rfm_df['F_score'] = rfm_df['F_score'].astype(int)
rfm_df['M_score'] = rfm_df['M_score'].astype(int)
rfm_df['RFM_score'] = rfm_df['R_score'] + rfm_df['F_score'] + rfm_df['M_score']

print("RFM Analysis Complete!")

RFM Analysis Complete!


In [9]:
# Create customer segments
def segment_customers(row):
    if row['RFM_score'] >= 12:
        return 'Champions'
    elif row['RFM_score'] >= 9:
        return 'Loyal Customers'
    elif row['RFM_score'] >= 7:
        return 'Potential Loyalists'
    elif row['RFM_score'] >= 5:
        return 'At Risk'
    else:
        return 'Lost'

rfm_df['customer_segment'] = rfm_df.apply(segment_customers, axis=1)
print("\nCustomer Segment Distribution:")
print(rfm_df['customer_segment'].value_counts())


Customer Segment Distribution:
customer_segment
Loyal Customers        2795
Champions              2371
Potential Loyalists    1703
At Risk                1374
Lost                    942
Name: count, dtype: int64


## 3. Customer Lifetime Value (CLV)

In [10]:
# Calculate CLV
customer_metrics = won_transactions.groupby('customer_id').agg({
    'net_amount': ['sum', 'mean', 'count'],
    'profit': 'sum',
    'transaction_date': ['min', 'max']
}).reset_index()

customer_metrics.columns = ['customer_id', 'total_revenue', 'avg_order_value', 
                              'purchase_frequency', 'total_profit', 
                              'first_purchase', 'last_purchase']

customer_metrics['lifespan_months'] = (
    (customer_metrics['last_purchase'] - customer_metrics['first_purchase']).dt.days / 30
).clip(lower=1)

customer_metrics['purchases_per_month'] = (
    customer_metrics['purchase_frequency'] / customer_metrics['lifespan_months']
)

customer_metrics['CLV'] = (
    customer_metrics['avg_order_value'] * customer_metrics['purchases_per_month'] * 24
)

print("CLV Calculation Complete!")
print("\nCLV Statistics:")
print(customer_metrics['CLV'].describe())

CLV Calculation Complete!

CLV Statistics:
count    9.185000e+03
mean     4.580812e+05
std      8.801586e+05
min      1.766051e+03
25%      7.520811e+04
50%      1.832561e+05
75%      4.598370e+05
max      1.398957e+07
Name: CLV, dtype: float64


## 4. Churn Prediction

In [11]:
# Define churn
churn_threshold_days = 180
max_date = won_transactions['transaction_date'].max()

customer_churn_data = customer_metrics.copy()
customer_churn_data['days_since_last_purchase'] = (
    (max_date - customer_churn_data['last_purchase']).dt.days
)
customer_churn_data['is_churned'] = (
    customer_churn_data['days_since_last_purchase'] > churn_threshold_days
).astype(int)

churn_analysis = customer_churn_data.merge(
    rfm_df[['customer_id', 'R_score', 'F_score', 'M_score', 'customer_segment']], 
    on='customer_id', how='left'
)

print(f"Churn Rate: {churn_analysis['is_churned'].mean()*100:.2f}%")

Churn Rate: 98.61%


In [12]:
# Train churn prediction model
features = ['avg_order_value', 'purchase_frequency', 'lifespan_months', 
            'purchases_per_month', 'R_score', 'F_score', 'M_score']

X = churn_analysis[features].fillna(0)
y = churn_analysis['is_churned']

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X, y)

churn_analysis['churn_probability'] = rf_model.predict_proba(X)[:, 1]

# Identify at-risk high-value customers
at_risk_customers = churn_analysis[
    (churn_analysis['churn_probability'] > 0.5) & 
    (churn_analysis['CLV'] > churn_analysis['CLV'].quantile(0.75))
].copy()

retention_revenue_opportunity = at_risk_customers['CLV'].sum() * 0.50
print(f"\nAt-Risk High-Value Customers: {len(at_risk_customers):,}")
print(f"Retention Opportunity: ${retention_revenue_opportunity:,.2f}")


At-Risk High-Value Customers: 2,295
Retention Opportunity: $1,581,306,310.56


## 5. Revenue Opportunity Identification

In [13]:
# Calculate all opportunities
current_annual_revenue = total_revenue
revenue_gap = current_annual_revenue * 0.15

# Opportunity 1: Retention
retention_opportunity = retention_revenue_opportunity

# Opportunity 2: Cross-sell
customer_products = won_transactions.groupby('customer_id')['product_id'].nunique().reset_index()
customer_products.columns = ['customer_id', 'unique_products']

cross_sell_candidates = customer_metrics.merge(customer_products, on='customer_id')
cross_sell_candidates = cross_sell_candidates[
    (cross_sell_candidates['unique_products'] <= 2) &
    (cross_sell_candidates['total_revenue'] > cross_sell_candidates['total_revenue'].median())
]

cross_sell_opportunity = len(cross_sell_candidates) * 0.20 * avg_deal_size

# Opportunity 3: Process Optimization
segment_performance = transactions_df.merge(customers_df[['customer_id', 'segment']], on='customer_id')
segment_win_rate = segment_performance.groupby('segment')['is_won'].mean()
worst_segment = segment_win_rate.idxmin()
worst_win_rate = segment_win_rate.min()

segment_deals = segment_performance[segment_performance['segment'] == worst_segment]
current_segment_revenue = segment_deals[segment_deals['is_won']]['net_amount'].sum()
process_optimization_opportunity = current_segment_revenue * 0.05 / worst_win_rate

# Opportunity 4: Market Expansion
state_performance = won_transactions.merge(customers_df[['customer_id', 'state']], on='customer_id')
state_revenue = state_performance.groupby('state')['net_amount'].sum()
underperforming_states = state_revenue[state_revenue < state_revenue.median()]
market_expansion_opportunity = underperforming_states.sum() * 0.25

total_opportunity = (retention_opportunity + cross_sell_opportunity + 
                     process_optimization_opportunity + market_expansion_opportunity)

print("="*60)
print("REVENUE OPPORTUNITY ANALYSIS")
print("="*60)
print(f"Revenue Gap: ${revenue_gap:,.2f}")
print(f"\n1. Customer Retention: ${retention_opportunity:,.2f}")
print(f"2. Cross-sell/Up-sell: ${cross_sell_opportunity:,.2f}")
print(f"3. Process Optimization: ${process_optimization_opportunity:,.2f}")
print(f"4. Market Expansion: ${market_expansion_opportunity:,.2f}")
print(f"\nTOTAL OPPORTUNITY: ${total_opportunity:,.2f}")
print("="*60)

REVENUE OPPORTUNITY ANALYSIS
Revenue Gap: $113,053,453.55

1. Customer Retention: $1,581,306,310.56
2. Cross-sell/Up-sell: $9,517,293.65
3. Process Optimization: $15,985,195.21
4. Market Expansion: $89,704,400.00

TOTAL OPPORTUNITY: $1,696,513,199.42


## 6. Sales Performance Analysis

In [14]:
# Rep performance
rep_performance = won_transactions.groupby('rep_id').agg({
    'transaction_id': 'count',
    'net_amount': ['sum', 'mean'],
    'profit': 'sum'
}).reset_index()

rep_performance.columns = ['rep_id', 'deals_closed', 'total_revenue', 'avg_deal_size', 'total_profit']
rep_performance = rep_performance.merge(sales_reps_df, on='rep_id')

# Product performance
product_performance = won_transactions.groupby('product_id').agg({
    'transaction_id': 'count',
    'net_amount': 'sum',
    'profit': 'sum'
}).reset_index()

product_performance.columns = ['product_id', 'units_sold', 'total_revenue', 'total_profit']
product_performance = product_performance.merge(products_df, on='product_id')

print("Performance Analysis Complete!")

Performance Analysis Complete!


## 7. Export Processed Data

In [17]:
# Customer insights
customer_insights = customers_df.merge(
    customer_metrics[['customer_id', 'total_revenue',
                      'avg_order_value', 'purchase_frequency', 'CLV']],
    on='customer_id', how='left'
).merge(
    rfm_df[['customer_id', 'recency', 'frequency',
            'monetary', 'RFM_score', 'customer_segment']],
    on='customer_id', how='left'
).merge(
    churn_analysis[['customer_id', 'churn_probability', 'is_churned']],
    on='customer_id', how='left'
).fillna(0)

# Enhanced transactions
transactions_enhanced = transactions_df.merge(
    customers_df[['customer_id', 'segment',
                  'industry', 'state']], on='customer_id'
).merge(
    products_df[['product_id', 'product_name', 'category']], on='product_id'
).merge(
    sales_reps_df[['rep_id', 'rep_name', 'region']], on='rep_id'
)

# Save all files
customer_insights.to_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/processed/customer_insights.csv', index=False)
transactions_enhanced.to_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/processed/transactions_enhanced.csv', index=False)
rep_performance.to_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/processed/sales_rep_performance.csv', index=False)
product_performance.to_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/processed/product_performance.csv', index=False)
at_risk_customers.to_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/processed/at_risk_customers.csv', index=False)
cross_sell_candidates.to_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/processed/cross_sell_candidates.csv', index=False)

# Create opportunities dataframe
opportunities = pd.DataFrame({
    'Opportunity': ['Customer Retention', 'Cross-sell/Up-sell', 'Sales Process Optimization', 'Market Expansion'],
    'Revenue_Impact': [retention_opportunity, cross_sell_opportunity, process_optimization_opportunity, market_expansion_opportunity],
    'Priority': ['High', 'High', 'Medium', 'Medium'],
    'Effort': ['Medium', 'Low', 'High', 'High'],
    'Timeline_Months': [3, 2, 6, 6]
})
opportunities.to_csv(
    'C:/Users/Asus/OneDrive/Desktop/sales-dashboard/data/processed/revenue_opportunities.csv', index=False)

print("\n" + "="*60)
print("ALL DATA EXPORTED FOR TABLEAU!")
print("="*60)
print("\nFiles saved in data/processed/")
print("Ready for Tableau import!")


ALL DATA EXPORTED FOR TABLEAU!

Files saved in data/processed/
Ready for Tableau import!
